# Notebook 6 — Train, Tune, Evaluate
Start with a simple baseline, train a model, tune on validation, then evaluate on the test set **once at the end**. Because this is an imbalanced classification problem, precision/recall/F1 and ROC-AUC are reported instead of relying on accuracy alone.

In [ ]:

import os, warnings
from pathlib import Path
import pandas as pd
import numpy as np
warnings.filterwarnings("ignore")

ART = Path("../artifacts")
ART.mkdir(exist_ok=True)
CHARTS = ART / "charts"
CHARTS.mkdir(exist_ok=True)

import joblib
from scipy import sparse
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, confusion_matrix, classification_report
)

Xtr = sparse.load_npz(ART/"05_X_train.npz")
Xv = sparse.load_npz(ART/"05_X_validation.npz")
Xte = sparse.load_npz(ART/"05_X_test.npz")
y_train = np.load(ART/"05_y_train.npy")
y_val = np.load(ART/"05_y_validation.npy")
y_test = np.load(ART/"05_y_test.npy")


In [ ]:

def metrics(y, pred, score=None):
    out = {
        "accuracy": accuracy_score(y,pred),
        "precision": precision_score(y,pred,zero_division=0),
        "recall": recall_score(y,pred,zero_division=0),
        "f1": f1_score(y,pred,zero_division=0)
    }
    if score is not None:
        out["roc_auc"] = roc_auc_score(y,score)
    return out

baseline = DummyClassifier(strategy="most_frequent")
baseline.fit(Xtr, y_train)
base_pred = baseline.predict(Xv)
baseline_metrics = metrics(y_val, base_pred)
print("Baseline:", baseline_metrics)


In [ ]:

# Validation tuning: small, transparent grid.
results = []
for C in [0.1, 0.5, 1.0, 2.0]:
    model = LogisticRegression(C=C, max_iter=1000, class_weight="balanced", solver="liblinear")
    model.fit(Xtr, y_train)
    pred = model.predict(Xv)
    score = model.predict_proba(Xv)[:,1]
    m = metrics(y_val, pred, score)
    m["C"] = C
    results.append(m)

val_results = pd.DataFrame(results).sort_values("f1", ascending=False)
display(val_results)
best_C = float(val_results.iloc[0]["C"])


In [ ]:

# Retrain selected model on train only; validation was used for model selection.
model = LogisticRegression(C=best_C, max_iter=1000, class_weight="balanced", solver="liblinear")
model.fit(Xtr, y_train)
val_pred = model.predict(Xv)
val_score = model.predict_proba(Xv)[:,1]
print("Selected model validation metrics:")
print(metrics(y_val, val_pred, val_score))


In [ ]:

# FINAL TEST TOUCH — only here.
test_pred = model.predict(Xte)
test_score = model.predict_proba(Xte)[:,1]
test_metrics = metrics(y_test, test_pred, test_score)
print("Final test metrics:", test_metrics)
print("\nClassification report:")
print(classification_report(y_test, test_pred, target_names=["On-time","Late"], zero_division=0))
print("\nConfusion matrix:")
print(confusion_matrix(y_test, test_pred))


In [ ]:

joblib.dump(model, ART/"06_trained_model.joblib")
val_results.to_csv(ART/"06_validation_results.csv", index=False)
pd.DataFrame([test_metrics]).to_csv(ART/"06_test_results.csv", index=False)
print("Saved trained model and results summary.")


In [ ]:

comparison = pd.DataFrame([
    {"model":"Most-frequent baseline", **baseline_metrics},
    {"model":"Tuned Logistic Regression", **test_metrics}
])
display(comparison)
comparison.to_csv(ART/"06_baseline_vs_model.csv", index=False)
